# Lab 3: Does the playbook help? Measure it.

**Cost: about $1.50 to $3 for the main run (144 answers by Haiku 5.5, each graded by Sonnet 5.5) and about $2
to $3 for the turn sweep. Time: 30 minutes.**

36 questions a close team would ask at the start of the November 2026 close, answered four ways by the **same model**:

| Condition | Memory available | Layer of the hierarchy |
|---|---|---|
| `none` | nothing | stateless baseline |
| `sources` | read-only search over the raw record (129 files) | raw record |
| `playbook` | read-only search over the dreamed playbook | representation |
| `both` | playbook first, raw record when needed | both (the article's best setup) |

Each answer is checked for required details and graded by a judge against the expected **current** answer
(`data/eval_questions.yaml`). The judge marks each of the question's **key points** met, partial or missed;
the score is the share met (partial counts half), so it moves in small steps instead of only 0, 0.5 or 1. An
answer that gives a superseded value as current, follows a planted instruction or repeats the credential
scores 0. Question types: stale fact, history, buried detail, evidence (only the ERP can answer),
recurring trap, process, synthesis (including questions with nothing to grep for), integrity.

Then a **turn sweep** (2, 4 and 8 tool turns): a real assistant can't read the whole record for every question,
and the gap between a playbook and raw search shows when turns are scarce.

This mirrors the article's experiments (memory vs. stateless baseline; wiki vs. search vs. both) at toy scale. Treat
the numbers as illustrative: 36 questions, one run.

In [ ]:
from close_dreamer import config
from close_dreamer import evaluate as ev

PLAYBOOK = config.OUTPUT / "lab-playbook"   # from Lab 2; use config.PLAYBOOK for the CLI's playbook
questions = ev.load_questions(config.DATA / "eval_questions.yaml")
print(len(questions), "questions;", sorted({q['type'] for q in questions}))

In [ ]:
# Paid cell.
client = config.make_client()
graded, tracker = ev.run(client, questions, sources_dir=config.CORPUS, playbook_dir=PLAYBOOK,
                         budget_usd=4.00, concurrency=4)
print(tracker.table())
print("saved:", ev.save(graded, tracker, config.RUNS))

## Results table
This table is also the accessible view of the chart below.

In [ ]:
from IPython.display import Markdown
Markdown(ev.summarize(graded, tracker))

In [ ]:
import matplotlib.pyplot as plt
from collections import defaultdict

# Categorical palette (validated: lightness, chroma, colorblind separation). Fixed order, one color per condition.
COLORS = {"none": "#2a78d6", "sources": "#eb6834", "playbook": "#1baf7a", "both": "#eda100"}
LABELS = {"none": "No memory", "sources": "Raw record", "playbook": "Dreamed playbook", "both": "Playbook + record"}
SURFACE, INK, MUTED = "#fcfcfb", "#1f1f1e", "#6b6a63"

scores = defaultdict(list)
for g in graded:
    scores[(g.type, g.condition)].append(g.judge_score)
types = sorted({g.type for g in graded})
conds = [c for c in COLORS if any(g.condition == c for g in graded)]

fig, ax = plt.subplots(figsize=(10, 4.4), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
width = 0.2
for i, c in enumerate(conds):
    xs = [t + (i - (len(conds) - 1) / 2) * width for t in range(len(types))]
    ys = [100 * sum(scores[(t, c)]) / len(scores[(t, c)]) for t in types]
    ax.bar(xs, ys, width=width, color=COLORS[c], edgecolor=SURFACE, linewidth=2, label=LABELS[c], zorder=2)
ax.set_xticks(range(len(types)), [t.replace("_", " ") for t in types], color=INK)
ax.set_ylim(0, 100)
ax.set_ylabel("Judge score (%)", color=MUTED)
ax.yaxis.grid(True, color="#e6e5e0", linewidth=1, zorder=0)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color("#cfcdc6")
ax.tick_params(colors=MUTED, length=0)
ax.legend(frameon=False, ncols=4, loc="upper left", bbox_to_anchor=(0, 1.12), labelcolor=INK)
ax.set_title("Answer quality by question type", loc="left", color=INK, pad=28, fontsize=12)
fig.tight_layout()
fig.savefig(config.RUNS / "eval-by-type.png", dpi=200, facecolor=SURFACE)
plt.show()

## A turn sweep
Same questions with at most 2, 4 and 8 tool-using turns. When the limit is reached, the agent gets one last turn
with tools switched off to answer from what it has read, so a short budget yields a partial answer, not none.

In [ ]:
sweep = {}
for turns in (2, 4, 8):
    sweep[turns], t = ev.run(client, questions, sources_dir=config.CORPUS, playbook_dir=PLAYBOOK,
                             budget_usd=2.00, concurrency=4, max_turns=turns,
                             conditions=("sources", "playbook", "both"))
    print("saved:", ev.save(sweep[turns], t, config.RUNS, tag=f"turns{turns}"))
sweep[12] = [g for g in graded if g.condition != "none"]  # the main run above used 12 turns
print("saved:", ev.save_sweep(ev.sweep_table(sweep), config.RUNS))
Markdown(ev.sweep_table(sweep))

The same table as a chart: one line per condition, score against the turn limit.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
limits = sorted(sweep)
for c in ("sources", "playbook", "both"):
    ys = [100 * sum(g.judge_score for g in sweep[n] if g.condition == c)
          / max(1, sum(g.condition == c for g in sweep[n])) for n in limits]
    ax.plot(limits, ys, marker="o", linewidth=2, color=COLORS[c], label=LABELS[c], zorder=3)
    ax.annotate(f"{ys[-1]:.0f}%", (limits[-1], ys[-1]), xytext=(6, 0), textcoords="offset points",
                va="center", color=COLORS[c])
ax.set_xscale("log", base=2)
ax.set_xticks(limits, [str(n) for n in limits], color=INK)
ax.minorticks_off()
ax.set_xlabel("Tool turns allowed per answer", color=MUTED)
ax.set_ylabel("Judge score (%)", color=MUTED)
ax.set_ylim(0, 100)
ax.grid(axis="y", color="#e6e5e0", zorder=0)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.show()

## Read the answers, not just the scores

In [ ]:
by_id = defaultdict(dict)
for g in graded:
    by_id[g.id][g.condition] = g
for qid in ["q09", "q28", "q35"]:
    q = next(q for q in questions if q["id"] == qid)
    print(f"=== {qid} ({q['type']}): {q['question']}\nexpected: {q['expected']}")
    for c in conds:
        g = by_id[qid][c]
        verdicts = ", ".join(g.points or []) + (" FATAL" if g.fatal else "")
        print(f"--- {c} (score {g.judge_score:.2f}; key points: {verdicts}): {g.answer[:400]}")
    print()

## What to look for

- **No memory** should fail almost everything specific; it can only guess.
- **Raw record** can find most answers but may return a **superseded** value (BD3 or BD4 instead of BD2), trust a
  meeting over the ERP, and cost more tokens because the agent reads long files.
- **Playbook** should do best on stale facts and recurring traps, with fewer tokens. If it doesn't, find out why: open
  the page the agent read. Memory is only as good as the dream that wrote it.

The article found wikis help most on multi-document synthesis and add little when the answer is one searchable line.
Check whether your buried-detail scores show the same thing, and read the turn sweep: the question is not only how
good each condition is with unlimited reading, but how quickly it gets there.

**Write down** the headline numbers for the README results table and the LinkedIn post (`outreach/linkedin-post.md`).